In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scorecardpy as sc
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
import pickle

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
df = pd.read_excel('../data/default_credit_card.xls', header=1)
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='object')

In [5]:
#ECL = PD（违约概率） × LGD（违约损失率） × EAD（违约风险暴露）
pay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
def status_map(x):
    if x<=0:
        return 'current'
    elif x==1:
        return 'M1' #1-30 days
    elif x==2:
        return 'M2' #31-60 days
    elif x==3:
        return 'M3' #61-90 days
    elif x == 4:
        return 'M4' #91-120 days(违约)
    elif x == 5:
        return 'M5' #121-150 days(违约)
    else:  # x >= 6
        return 'M6' #151+ days(严重违约)
for col in pay_cols:
    df[col+"_status"] = df[col].apply(status_map)

In [9]:
# 统计上月(PAY_2_status)到当月(PAY_0_status)的迁徙
transition_count = {}
states = ["current","M1","M2","M3","M4","M5","M6"]
trans_matrix = pd.DataFrame(0, index=states, columns=states)

for idx, row in df.iterrows():
    s_from = row["PAY_2_status"]
    s_to = row["PAY_0_status"]
    trans_matrix.loc[s_from, s_to] += 1
trans_matrix

,current,M1,M2,M3,M4,M5,M6
current,22735,1836,991,0,0,0,0
M1,0,28,0,0,0,0,0
M2,392,1672,1591,272,0,0,0
M3,47,109,71,41,58,0,0
M4,5,32,14,8,15,25,0
M5,3,7,0,1,3,0,11
M6,0,4,0,0,0,1,28


In [10]:
# 转换为概率矩阵
trans_prob = trans_matrix.div(trans_matrix.sum(axis=1), axis=0)
trans_prob

,current,M1,M2,M3,M4,M5,M6
current,0.889406,0.071825,0.038768,0.000000,0.000000,0.000000,0.000000
M1,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
M2,0.099822,0.425770,0.405144,0.069264,0.000000,0.000000,0.000000
M3,0.144172,0.334356,0.217791,0.125767,0.177914,0.000000,0.000000
M4,0.050505,0.323232,0.141414,0.080808,0.151515,0.252525,0.000000
M5,0.120000,0.280000,0.000000,0.040000,0.120000,0.000000,0.440000
M6,0.000000,0.121212,0.000000,0.000000,0.000000,0.030303,0.848485
